In [1]:
import hashlib
import json
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import roc_auc_score

project_path = Path.cwd()
if project_path.name == "notebooks":
    project_path = project_path.parent
ext_path = project_path / "data" / "external"
out_path = project_path / "docs" / "overfitting" / "unseen"
out_path.mkdir(parents=True, exist_ok=True)

C_LAB, C_EXP, C_REF = "#2a78d6", "#eb6834", "#6b7280"

In [2]:
manifest = json.loads((ext_path / "unseen_v1.manifest.json").read_text(encoding="utf-8"))
csv_file = ext_path / manifest["file"]
digest = hashlib.sha256(csv_file.read_bytes()).hexdigest()
assert digest == manifest["sha256"], "unseen_v1.csv ka ndryshuar pas ngrirjes - vlerësimi ndalohet."
print("Hash OK:", digest)
print("Ngrirë më:", manifest["frozen_on"], "| Rreshta:", manifest["rows"])
print("Verifikimi:", manifest["verification"])

unseen = pd.read_csv(csv_file, keep_default_na=False)
unseen.groupby(["evaluation_role", "category"]).size().rename("raste").to_frame()

Hash OK: f738b1508733747a99ccdc401f78f18de9008d0addc3261e2a78548454511cc7
Ngrirë më: 2026-11-12 | Rreshta: 118
Verifikimi: Labels are proposed, NOT clinically verified. This file is not evidence of clinical accuracy.


raste
evaluation_role category           
exploratory     ambiguous         8
                out_of_scope     14
                overlapping      10
                unrelated         6
labelled        everyday         24
                negation          8
                typical          48

In [3]:
model = joblib.load(project_path / "models" / "svm_tuned_validation.joblib")
tfidf = model.named_steps["tfidf"]
vocab = tfidf.vocabulary_
analyzer = tfidf.build_analyzer()
print("Modeli:", model.named_steps["classifier"], "| Fjalori:", len(vocab), "fjalë")

def describe(texts):
    scores = model.decision_function(texts)
    top2 = np.sort(scores, axis=1)[:, -2:]
    coverage = []
    for t in texts:
        tokens = analyzer(t)
        coverage.append(np.mean([tok in vocab for tok in tokens]) if tokens else 0.0)
    return pd.DataFrame({
        "predicted": model.classes_[scores.argmax(axis=1)],
        "top_score": top2[:, 1],
        "margin": top2[:, 1] - top2[:, 0],
        "vocab_coverage": coverage,
    })

pred = pd.concat([unseen.reset_index(drop=True), describe(unseen["text"].tolist())], axis=1)
acceptable = pred["acceptable_labels"].str.split(";")
pred["in_acceptable"] = pd.array([(p in a) if t != "none" else pd.NA
                                  for p, a, t in zip(pred["predicted"], acceptable, pred["label_type"])], dtype="boolean")

val_df = pd.read_csv(project_path / "data" / "processed" / "validation.csv")
val_ref = describe(val_df["text"].tolist())
val_ref["correct"] = val_ref["predicted"].values == val_df["label"].values
print("Validation accuracy (kontroll):", round(val_ref["correct"].mean() * 100, 2), "%")

pred.to_csv(out_path / "01_unseen_predictions.csv", index=False)
pred[["id", "category", "text", "acceptable_labels", "predicted", "top_score", "margin", "vocab_coverage"]].head(8)

Modeli: LinearSVC(C=10, random_state=42) | Fjalori: 1382 fjalë
Validation accuracy (kontroll): 99.44 %


,id,category,text,acceptable_labels,predicted,top_score,margin,vocab_coverage
0,U001,typical,Lots of red pimples and blackheads on my foreh...,Acne,Acne,-0.228647,0.572362,0.833333
1,U002,typical,Painful spots keep popping up on my back and s...,Acne,Fungal infection,-0.551885,0.157327,0.812500
2,U003,typical,My knees and fingers ache and feel stiff for t...,Arthritis,Psoriasis,-0.108021,0.567343,1.000000
3,U004,typical,It hurts to grip things because my wrist joint...,Arthritis,Arthritis,0.181623,0.680637,0.842105
4,U005,typical,I get tightness in my chest and a whistling so...,Bronchial Asthma,allergy,-0.164348,0.040146,0.882353
5,U006,typical,Cold air makes me short of breath and I start ...,Bronchial Asthma,allergy,-0.521610,0.104041,0.875000
6,U007,typical,"The back of my neck is stiff and sore, and the...",Cervical spondylosis,gastroesophageal reflux disease,-0.171656,0.320642,0.950000
7,U008,typical,"Turning my head is painful, my neck clicks, an...",Cervical spondylosis,drug reaction,-0.503147,0.222705,0.882353


In [4]:
def wilson(k, n, z=norm.ppf(0.975)):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    centre = (p + z**2 / (2 * n)) / (1 + z**2 / n)
    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / (1 + z**2 / n)
    return centre - half, centre + half

labelled = pred[pred["evaluation_role"] == "labelled"]
rows = []
for name, part in list(labelled.groupby("category", sort=False)) + [("të gjitha", labelled)]:
    k, n = int(part["in_acceptable"].sum()), len(part)
    lo, hi = wilson(k, n)
    rows.append({"Kategoria": name, "Raste": n, "Të sakta": k, "Përputhja (%)": round(k / n * 100, 2),
                 "Wilson 95% (%)": f"{lo * 100:.1f} – {hi * 100:.1f}",
                 "Mbulimi mesatar i fjalorit (%)": round(part["vocab_coverage"].mean() * 100, 1)})
labelled_summary = pd.DataFrame(rows)
labelled_summary.to_csv(out_path / "02_labelled_agreement.csv", index=False)
labelled_summary

,Kategoria,Raste,Të sakta,Përputhja (%),Wilson 95% (%),Mbulimi mesatar i fjalorit (%)
0,typical,48,28,58.33,44.3 – 71.2,90.2
1,everyday,24,12,50.00,31.4 – 68.6,88.2
2,negation,8,5,62.50,30.6 – 86.3,95.0
3,të gjitha,80,45,56.25,45.3 – 66.6,90.1


In [5]:
per_class = (labelled.groupby("proposed_label")["in_acceptable"]
             .agg(["sum", "count"]).rename(columns={"sum": "Të sakta", "count": "Raste"}))
per_class["Përputhja (%)"] = (per_class["Të sakta"] / per_class["Raste"] * 100).round(1)
per_class = per_class.sort_values(["Përputhja (%)", "Raste"])
per_class.to_csv(out_path / "03_labelled_per_class.csv")
per_class.head(12)

,Të sakta,Raste,Përputhja (%)
proposed_label,,,
Bronchial Asthma,0,3,0.0
Cervical spondylosis,0,3,0.0
Hypertension,0,3,0.0
drug reaction,0,3,0.0
Migraine,0,4,0.0
Chicken pox,1,3,33.3
Dimorphic Hemorrhoids,1,3,33.3
Fungal infection,1,3,33.3
Typhoid,1,3,33.3


In [6]:
clf = model.named_steps["classifier"]
feature_names = np.array(tfidf.get_feature_names_out())

def top_words(text, cls, k=4):
    x = tfidf.transform([text])
    contrib = x.multiply(clf.coef_[list(clf.classes_).index(cls)]).toarray().ravel()
    idx = [i for i in contrib.argsort()[::-1][:k] if contrib[i] > 0]
    return ", ".join(feature_names[idx])

errors = labelled[~labelled["in_acceptable"].astype(bool)].copy()
errors["fjalët kryesore"] = [top_words(t, p) for t, p in zip(errors["text"], errors["predicted"])]
errors = errors[["id", "category", "text", "proposed_label", "predicted", "margin", "vocab_coverage", "fjalët kryesore"]]
errors.to_csv(out_path / "04_labelled_errors.csv", index=False)
pd.set_option("display.max_colwidth", 90)
errors

,id,category,text,proposed_label,predicted,margin,vocab_coverage,fjalët kryesore
1,U002,typical,"Painful spots keep popping up on my back and shoulders, some of them with white heads.",Acne,Fungal infection,0.157327,0.812500,"some, popping, spots, of"
2,U003,typical,"My knees and fingers ache and feel stiff for the first hour after I wake up, and the j...",Arthritis,Psoriasis,0.567343,1.000000,"knees, fingers, joints, after"
4,U005,typical,"I get tightness in my chest and a whistling sound when I breathe out, especially at ni...",Bronchial Asthma,allergy,0.040146,0.882353,"running, night, breathe, get"
5,U006,typical,Cold air makes me short of breath and I start wheezing and coughing until I use my inh...,Bronchial Asthma,allergy,0.104041,0.875000,"wheezing, short, start, me"
6,U007,typical,"The back of my neck is stiff and sore, and the pain spreads into my shoulder and down ...",Cervical spondylosis,gastroesophageal reflux disease,0.320642,0.950000,"arm, spreads, down, the"
7,U008,typical,"Turning my head is painful, my neck clicks, and I sometimes feel dizzy with tingling i...",Cervical spondylosis,drug reaction,0.222705,0.882353,"hands, sometimes, dizzy, in"
9,U010,typical,"Red spots turned into fluid-filled blisters that itch like crazy, and they are spreadi...",Chicken pox,Impetigo,0.257147,1.000000,"blisters, fluid, spreading, turned"
10,U011,typical,"Stuffy nose, constant sneezing and a scratchy throat that started two days ago, with a...",Common Cold,Impetigo,0.052031,1.000000,"nose, ago, two, days"
13,U014,typical,"After the fever started I got a red rash on my arms, intense muscle pain and I feel na...",Dengue,Chicken pox,0.009886,1.000000,"red, fever, rash, intense"
15,U016,typical,"I feel a painful lump near my bottom and going to the toilet is uncomfortable, sometim...",Dimorphic Hemorrhoids,Impetigo,0.018780,0.875000,"near, painful, the, to"


In [7]:
fig, ax = plt.subplots(figsize=(7, 3.4))
plot_df = labelled_summary[labelled_summary["Kategoria"] != "të gjitha"]
bars = ax.barh(plot_df["Kategoria"], plot_df["Përputhja (%)"], color=C_LAB, height=0.55)
val_acc = val_ref["correct"].mean() * 100
ax.axvline(val_acc, color=C_REF, linestyle="--", linewidth=1.2)
ax.annotate(f"Validation {val_acc:.1f}%", xy=(val_acc, 1.0), xycoords=("data", "axes fraction"),
            xytext=(-4, -12), textcoords="offset points", ha="right", fontsize=9, color=C_REF)
for b, (_, r) in zip(bars, plot_df.iterrows()):
    ax.text(b.get_width() - 1, b.get_y() + b.get_height() / 2, f"{r['Përputhja (%)']:.1f}%  ({r['Të sakta']}/{r['Raste']})",
            va="center", ha="right", color="white", fontsize=9)
ax.set_xlim(0, 100)
ax.set_xlabel("Përputhja me etiketën e propozuar (%)")
ax.set_title("Unseen v1 (sintetik, i paverifikuar klinikisht)")
ax.invert_yaxis()
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(out_path / "02_labelled_agreement.png", dpi=150)

In [8]:
overlap = pred[pred["category"] == "overlapping"]
print(f"Overlapping: parashikimi brenda klasave të mundshme në {int(overlap['in_acceptable'].sum())}/{len(overlap)} raste")
overlap[["id", "text", "acceptable_labels", "predicted", "margin"]]

Overlapping: parashikimi brenda klasave të mundshme në 9/10 raste


,id,text,acceptable_labels,predicted,margin
80,U081,"I have a high fever, a bad headache and my whole body aches.",Dengue;Malaria;Typhoid,Dengue,0.094794
81,U082,"I have a cough, some fever and my chest hurts a little.",Pneumonia;Common Cold;Bronchial Asthma,Pneumonia,0.145561
82,U083,I have an itchy rash on my arms.,Fungal infection;drug reaction;allergy;Psoriasis;Chicken pox,Psoriasis,0.030472
83,U084,I have burning pain in my upper stomach and chest after meals.,gastroesophageal reflux disease;peptic ulcer disease,peptic ulcer disease,0.210178
84,U085,I keep sneezing and my nose is running.,Common Cold;allergy,Common Cold,0.131358
85,U086,I get headaches and feel dizzy.,Hypertension;Migraine;Cervical spondylosis,allergy,0.306403
86,U087,I need to urinate much more often than before.,diabetes;urinary tract infection,diabetes,0.271514
87,U088,I have a fever and a red rash appeared on my body.,Dengue;Chicken pox;drug reaction,Chicken pox,0.110172
88,U089,My stomach hurts and I have lost my appetite and feel weak.,Jaundice;peptic ulcer disease;Typhoid,peptic ulcer disease,0.094084
89,U090,My neck and shoulders are stiff and painful.,Cervical spondylosis;Arthritis,Arthritis,0.178248


In [9]:
no_class = pred[pred["label_type"] == "none"]
no_class[["id", "category", "text", "predicted", "top_score", "margin", "vocab_coverage"]].sort_values(["category", "margin"])

,id,category,text,predicted,top_score,margin,vocab_coverage
90,U091,ambiguous,I just don't feel well lately.,diabetes,-0.499165,0.089310,1.000000
95,U096,ambiguous,"I have pain sometimes, it comes and goes.",allergy,-0.547666,0.096834,1.000000
93,U094,ambiguous,I feel hot and weak today.,diabetes,-0.443345,0.113033,1.000000
92,U093,ambiguous,Something is wrong with my stomach.,Dengue,-0.635151,0.133993,1.000000
91,U092,ambiguous,"I'm tired and a bit achy, nothing specific.",Common Cold,-0.626260,0.156910,0.666667
96,U097,ambiguous,I feel off and can't really explain it.,Chicken pox,-0.244484,0.274161,1.000000
97,U098,ambiguous,My body hurts and I am worried.,drug reaction,-0.532483,0.275581,1.000000
94,U095,ambiguous,My skin looks strange in some places.,Fungal infection,0.077234,0.518233,1.000000
111,U112,out_of_scope,"My skin is red, hot and painful after a day at the beach without sunscreen.",peptic ulcer disease,-0.742841,0.050311,0.857143
107,U108,out_of_scope,"I twisted my ankle playing football, it is swollen and bruised and I cannot put weight...",Psoriasis,-0.497659,0.073861,0.687500


In [10]:
dist = (no_class.groupby("category")["predicted"].value_counts().rename("raste").reset_index()
        .sort_values(["category", "raste"], ascending=[True, False]))
dist.to_csv(out_path / "05_no_class_prediction_distribution.csv", index=False)
dist.groupby("category").head(4)

,category,predicted,raste
0,ambiguous,diabetes,2
1,ambiguous,Common Cold,1
2,ambiguous,Dengue,1
3,ambiguous,Fungal infection,1
7,out_of_scope,peptic ulcer disease,2
8,out_of_scope,diabetes,2
9,out_of_scope,drug reaction,2
10,out_of_scope,Impetigo,2
17,unrelated,Chicken pox,1
18,unrelated,diabetes,1


In [11]:
groups = {
    "Validation": val_ref,
    "Unseen me etiketë": pred[pred["evaluation_role"] == "labelled"],
    "Overlapping": pred[pred["category"] == "overlapping"],
    "Ambiguous": pred[pred["category"] == "ambiguous"],
    "Out of scope": pred[pred["category"] == "out_of_scope"],
    "Unrelated": pred[pred["category"] == "unrelated"],
}
score_rows = []
for name, g in groups.items():
    score_rows.append({"Grupi": name, "Raste": len(g),
                       "Margin mesatarja": round(g["margin"].median(), 3),
                       "Top score mesatarja": round(g["top_score"].median(), 3),
                       "Mbulimi i fjalorit (%)": round(g["vocab_coverage"].median() * 100, 1)})
score_table = pd.DataFrame(score_rows).rename(columns={"Margin mesatarja": "Margin (mediana)",
                                                       "Top score mesatarja": "Top score (mediana)"})

lab = pred[pred["evaluation_role"] == "labelled"]
y = np.r_[np.ones(len(lab)), np.zeros(len(no_class))]
auroc = {
    "margin": roc_auc_score(y, np.r_[lab["margin"], no_class["margin"]]),
    "top_score": roc_auc_score(y, np.r_[lab["top_score"], no_class["top_score"]]),
    "vocab_coverage": roc_auc_score(y, np.r_[lab["vocab_coverage"], no_class["vocab_coverage"]]),
}
val_p05 = np.percentile(val_ref["margin"], 5)
share_low = {name: round((g["margin"] < val_p05).mean() * 100, 1) for name, g in groups.items()}
score_table["Margin < P5 e Validation (%)"] = score_table["Grupi"].map(share_low)
score_table.to_csv(out_path / "06_score_comparison.csv", index=False)
print("AUROC (me etiketë vs pa klasë):", {k: round(v, 3) for k, v in auroc.items()})
print(f"P5 e margin-it në Validation: {val_p05:.3f} (vetëm referencë përshkruese)")
score_table

AUROC (me etiketë vs pa klasë): {'margin': 0.668, 'top_score': 0.772, 'vocab_coverage': 0.671}
P5 e margin-it në Validation: 0.332 (vetëm referencë përshkruese)


,Grupi,Raste,Margin (mediana),Top score (mediana),Mbulimi i fjalorit (%),Margin < P5 e Validation (%)
0,Validation,178,1.458,0.698,100.0,5.1
1,Unseen me etiketë,80,0.396,-0.169,90.9,45.0
2,Overlapping,10,0.138,-0.158,100.0,100.0
3,Ambiguous,8,0.145,-0.516,100.0,87.5
4,Out of scope,14,0.308,-0.386,83.5,64.3
5,Unrelated,6,0.140,-0.417,61.3,100.0


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
names = list(groups)
colors = [C_REF, C_LAB, C_LAB, C_EXP, C_EXP, C_EXP]
for ax, col, title in [(axes[0], "margin", "Margin (klasa 1 - klasa 2)"),
                       (axes[1], "vocab_coverage", "Mbulimi i fjalorit")]:
    data = [groups[n][col] * (100 if col == "vocab_coverage" else 1) for n in names]
    bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.55, medianprops={"color": "#111827"})
    for patch, c in zip(bp["boxes"], colors):
        patch.set_facecolor(c)
        patch.set_alpha(0.55)
    ax.set_yticks(range(1, len(names) + 1), names)
    ax.invert_yaxis()
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
axes[1].set_xlabel("% e fjalëve në fjalor")
axes[0].set_xlabel("decision score")
fig.suptitle("Validation vs Unseen v1 (sintetik, i paverifikuar klinikisht)")
fig.tight_layout()
fig.savefig(out_path / "03_score_distributions.png", dpi=150)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6384\2316152677.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.55, medianprops={"color": "#111827"})
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_6384\2316152677.py:7: MatplotlibDeprecationWarning: vert: bool was deprecated in Matplotlib 3.11 and will be removed in 3.13. Use orientation: {'vertical', 'horizontal'} instead.
  bp = ax.boxplot(data, vert=False, patch_artist=True, widths=0.55, medianprops={"color": "#111827"})


In [13]:
summary = {
    "dataset": {"file": manifest["file"], "sha256": digest, "rows": int(len(pred)),
                "source": "synthetic_ai", "verification": "not_clinically_verified"},
    "model": "models/svm_tuned_validation.joblib (unchanged)",
    "labelled_agreement": {r["Kategoria"]: {"n": int(r["Raste"]), "correct": int(r["Të sakta"]),
                                            "agreement_pct": float(r["Përputhja (%)"]), "wilson95_pct": r["Wilson 95% (%)"]}
                           for _, r in labelled_summary.iterrows()},
    "validation_accuracy_reference_pct": round(float(val_acc), 2),
    "overlapping_in_plausible_set": f"{int(overlap['in_acceptable'].sum())}/{len(overlap)}",
    "no_class_cases": int(len(no_class)),
    "no_class_note": "The model always returns one of the 24 classes; it has no 'unknown' option.",
    "auroc_labelled_vs_no_class": {k: round(float(v), 3) for k, v in auroc.items()},
    "median_margin": dict(zip(score_table["Grupi"], score_table["Margin (mediana)"].astype(float))),
    "median_vocab_coverage_pct": dict(zip(score_table["Grupi"], score_table["Mbulimi i fjalorit (%)"].astype(float))),
    "labelled_errors": int(len(errors)),
    "disclaimer": "Synthetic, AI-written cases with proposed labels; not evidence of clinical accuracy.",
}
(out_path / "unseen_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
summary

{'dataset': {'file': 'unseen_v1.csv',
  'sha256': 'f738b1508733747a99ccdc401f78f18de9008d0addc3261e2a78548454511cc7',
  'rows': 118,
  'source': 'synthetic_ai',
  'verification': 'not_clinically_verified'},
 'model': 'models/svm_tuned_validation.joblib (unchanged)',
 'labelled_agreement': {'typical': {'n': 48,
   'correct': 28,
   'agreement_pct': 58.33,
   'wilson95_pct': '44.3 – 71.2'},
  'everyday': {'n': 24,
   'correct': 12,
   'agreement_pct': 50.0,
   'wilson95_pct': '31.4 – 68.6'},
  'negation': {'n': 8,
   'correct': 5,
   'agreement_pct': 62.5,
   'wilson95_pct': '30.6 – 86.3'},
  'të gjitha': {'n': 80,
   'correct': 45,
   'agreement_pct': 56.25,
   'wilson95_pct': '45.3 – 66.6'}},
 'validation_accuracy_reference_pct': 99.44,
 'overlapping_in_plausible_set': '9/10',
 'no_class_cases': 28,
 'no_class_note': "The model always returns one of the 24 classes; it has no 'unknown' option.",
 'auroc_labelled_vs_no_class': {'margin': 0.668,
  'top_score': 0.772,
  'vocab_coverage': 0

Interpretimi i plotë i rezultateve është në `docs/overfitting/README.md`.